# Dataset 8: EmpatheticDialogues — Data Cleaning & Preprocessing
**Model target:** Model 5 — Mistral 7B QLoRA (combined with CounselChat)  
**Source:** `facebook/empathetic_dialogues` (HuggingFace Datasets Hub)  
**Task:** Instruction fine-tuning for empathic conversational response generation  

---
## 8.1 Dataset Description
EmpatheticDialogues (Rashkin et al., 2019, Facebook AI Research) contains 25,000 multi-turn conversations where one person describes an emotional situation and another responds with empathy. Conversations are grounded in 32 emotion categories.

**Why combine with CounselChat for Model 5?**  
CounselChat gives **clinical precision** — real therapist technique. But 930 examples is a small training set for a 7B model. EmpatheticDialogues gives **conversational volume** (25k conversations) and **natural empathic tone**. Together, the model learns to be both clinically appropriate (from CounselChat) and naturally warm (from EmpatheticDialogues).

**Ratio decision:** 10k EmpatheticDialogues to 847 CounselChat (~12:1). This ratio ensures clinical technique is not drowned out by conversational style.

In [1]:
import re, json, random, unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from datasets import load_dataset, Dataset, DatasetDict, concatenate_datasets, load_from_disk

sns.set_theme(style='whitegrid')
random.seed(42)
np.random.seed(42)
print('Ready.')

Ready.


In [7]:
raw = load_dataset('facebook/empathetic_dialogues', trust_remote_code=True)
print(raw)
df = raw['train'].to_pandas()
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
print(f'Nulls: {df.isnull().sum().to_dict()}')
print(f'\nSample row:')
print(df.iloc[0])

Using the latest cached version of the module from C:\Users\M S I\.cache\huggingface\modules\datasets_modules\datasets\facebook--empathetic_dialogues\09bbeed3882a67db98c73952fb3c1c9a85af83dc78f81454c2454382fd03f6cf (last modified on Fri May 22 12:44:46 2026) since it couldn't be found locally at facebook/empathetic_dialogues, or remotely on the Hugging Face Hub.


DatasetDict({
    train: Dataset({
        features: ['conv_id', 'utterance_idx', 'context', 'prompt', 'speaker_idx', 'utterance', 'selfeval', 'tags'],
        num_rows: 76673
    })
    validation: Dataset({
        features: ['conv_id', 'utterance_idx', 'context', 'prompt', 'speaker_idx', 'utterance', 'selfeval', 'tags'],
        num_rows: 12030
    })
    test: Dataset({
        features: ['conv_id', 'utterance_idx', 'context', 'prompt', 'speaker_idx', 'utterance', 'selfeval', 'tags'],
        num_rows: 10943
    })
})
Shape: (76673, 8)
Columns: ['conv_id', 'utterance_idx', 'context', 'prompt', 'speaker_idx', 'utterance', 'selfeval', 'tags']
Nulls: {'conv_id': 0, 'utterance_idx': 0, 'context': 0, 'prompt': 0, 'speaker_idx': 0, 'utterance': 0, 'selfeval': 0, 'tags': 0}

Sample row:
conv_id                                               hit:0_conv:1
utterance_idx                                                    1
context                                                sentimental
prom

In [8]:
# ── EDA ──────────────────────────────────────────────────────────
df['context_words']   = df['context'].apply(lambda x: len(str(x).split()))
df['utterance_words'] = df['utterance'].apply(lambda x: len(str(x).split()))

print('Context (prompt) word count stats:')
print(df['context_words'].describe())
print('\nUtterance (response) word count stats:')
print(df['utterance_words'].describe())

# Emotion distribution
emotion_counts = df['emotion'].value_counts()
print(f'\n{len(emotion_counts)} unique emotions in dataset.')

# Problem: very short responses
short_responses = (df['utterance_words'] < 8).sum()
print(f'\nResponses < 8 words  : {short_responses:,} ({short_responses/len(df)*100:.1f}%)')
print('These are acknowledgements, not therapeutic responses.')

# Show short response examples
print('\nSample short responses (< 8 words):')
for txt in df[df['utterance_words'] < 8]['utterance'].head(5):
    print(f'  → "{txt}"')

Context (prompt) word count stats:
count    76673.0
mean         1.0
std          0.0
min          1.0
25%          1.0
50%          1.0
75%          1.0
max          1.0
Name: context_words, dtype: float64

Utterance (response) word count stats:
count    76673.000000
mean        16.379273
std        161.493414
min          1.000000
25%          8.000000
50%         12.000000
75%         17.000000
max      17637.000000
Name: utterance_words, dtype: float64


KeyError: 'emotion'

In [ ]:
# ── Cleaning function ────────────────────────────────────────────
RE_URL   = re.compile(r'https?://\S+|www\.\S+')
RE_SPACE = re.compile(r'[ \t]+')

def clean_empathetic(text, min_words=3, max_words=200):
    """
    Clean EmpatheticDialogues text.

    min_words=3 for context (prompts can be short situational descriptions)
    min_words applied separately to utterances (responses) at 12 words.
    """
    if not isinstance(text, str) or not text.strip():
        return None
    text = unicodedata.normalize('NFKC', text)
    # Replace underscore-escaped spaces (dataset artifact: 'I_feel_sad')
    text = text.replace('_comma_', ',')  # Dataset-specific artifact
    text = RE_URL.sub(' ', text)
    text = RE_SPACE.sub(' ', text).strip()
    words = text.split()
    if len(words) < min_words:
        return None
    return ' '.join(words[:max_words])

# Test
print(clean_empathetic('I_comma_ feel_comma_ so_comma_ lost'))

In [ ]:
# ── Apply cleaning ───────────────────────────────────────────────
from tqdm.auto import tqdm

rows = []
stats = {'kept':0,'short_context':0,'short_utterance':0,'null':0}

for split_name in ['train','validation','test']:
    for ex in tqdm(raw[split_name], desc=split_name):
        context   = clean_empathetic(str(ex.get('context','')), min_words=3)
        utterance = clean_empathetic(str(ex.get('utterance','')), min_words=12)

        if context is None:
            stats['null' if not str(ex.get('context','')).strip() else 'short_context']+=1
            continue
        if utterance is None:
            stats['short_utterance']+=1
            continue

        rows.append({
            'text': f'<s>[INST] {context} [/INST] {utterance}</s>',
            'context': context,
            'response': utterance,
            'emotion': str(ex.get('emotion','neutral')),
            'source': 'empathetic_dialogues',
        })
        stats['kept']+=1

print(f'\nStats: {stats}')
print(f'Total kept: {stats["kept"]:,}')

In [ ]:
# ── Sampling: 10k from full cleaned set ──────────────────────────
print('Why sample 10,000 not all?')
print('Full dataset: ~18k after cleaning')
print('CounselChat: 847 examples')
print('Ratio goal: ~12:1 empathetic:clinical')
print('10,000:847 = 11.8:1 ✓ close to target')
print()
print('If we used all 18k:')
print('  18,000:847 = 21:1 — clinical technique overwhelmed')
print('  Model would be warm but not therapeutic')
print()
print('seed=42 ensures SAME 10k every run (reproducibility)')

sampled = random.sample(rows, min(10000, len(rows)))
print(f'\nSampled: {len(sampled):,} examples')

# Emotion distribution in sample
emotion_dist = Counter(r['emotion'] for r in sampled)
print(f'Unique emotions in sample: {len(emotion_dist)}')
print('Top 10:')
for e, c in emotion_dist.most_common(10):
    print(f'  {e:20s}: {c}')

In [ ]:
# ── Combine with CounselChat and save ────────────────────────────
# Load cleaned CounselChat (from notebook 06)
try:
    cc = load_from_disk('data/cleaned/counselchat')
    cc_rows = [{'text': ex['text'], 'source': 'counselchat',
                'emotion': 'therapy'}
               for ex in cc['train']]
    print(f'CounselChat train examples: {len(cc_rows)}')
except:
    cc_rows = []
    print('CounselChat not found — run notebook 06 first')

all_rows = sampled + cc_rows
random.shuffle(all_rows)
print(f'Combined dataset: {len(all_rows):,} examples')
print(f'  EmpatheticDialogues: {len(sampled):,}')
print(f'  CounselChat:         {len(cc_rows):,}')

full_df = pd.DataFrame(all_rows)
n = len(full_df)
train_df = full_df.iloc[:int(n*0.85)]
val_df   = full_df.iloc[int(n*0.85):int(n*0.92)]
test_df  = full_df.iloc[int(n*0.92):]

ds = DatasetDict({
    'train':      Dataset.from_pandas(train_df, preserve_index=False),
    'validation': Dataset.from_pandas(val_df,   preserve_index=False),
    'test':       Dataset.from_pandas(test_df,  preserve_index=False),
})
ds.save_to_disk('data/cleaned/model5_combined')
print(f'\nModel 5 training dataset saved.')
print(f'  train: {len(train_df)}, val: {len(val_df)}, test: {len(test_df)}')
print('\n✓ EmpatheticDialogues preprocessing complete.')

# Push to HuggingFace Hub (replace with your username)
ds.push_to_hub('AmiruMallawarachchi/mindlens-model5-combined-cleaned')
print('\n✓ Model 5 combined dataset pushed to HuggingFace Hub.')

## 8.7 Preprocessing Summary

| Step | Problem | Fix | Reason |
|------|---------|-----|--------|
| 1 | `_comma_` artifact | Replace with `,` | Dataset encoding issue |
| 2 | Short responses <12 words | Filter | Acknowledgements ≠ therapeutic responses |
| 3 | Multi-turn structure | Take (context, utterance) pairs | SFTTrainer needs single instruction-response pairs |
| 4 | 18k → 10k sampling | Random sample (seed=42) | Balance 12:1 ratio with CounselChat |
| 5 | Combine with CounselChat | Concatenate + shuffle | Clinical precision + empathic volume |

**Final Model 5 dataset:** ~10,847 training pairs (10,000 EmpatheticDialogues + 847 CounselChat). Both sources formatted as Mistral instruction pairs. Saved to `data/cleaned/model5_combined`.